# 2 delay ponctuality

Converted from a Marimo HTML export to a Jupyter Notebook (`.ipynb`).
Code order follows the original Marimo app.


In [ ]:
import marimo as mo


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, time, timedelta
import datetime as dt
from scipy import stats
import warnings
import geopandas as gpd
from shapely.geometry import Point
warnings.filterwarnings('ignore')

from tqdm import tqdm
import gc


PUNCTUALITY ANALYSIS FOR STIB-MIVB

===================================

Punctuality measures if vehicles arrive at stops at their scheduled time.
This is relevant for INFREQUENT lines (headway > 12 minutes) where passengers
plan their trips based on specific arrival times.

Key Metrics:
- Delay: Actual arrival time - Scheduled arrival time (in seconds/minutes)
- On-Time Performance: % of arrivals within acceptable threshold (e.g., ±1 min)
- Early/Late Distribution
- Delay patterns by line, stop, time of day, day of week


### Load tables


In [ ]:
stops = pd.read_csv("data/feed/stops.csv" , low_memory=False)
stops.drop(columns=['location_type'], inplace=True)
stops


In [ ]:
df_punctuality = pd.read_parquet("data/stib_all_punctuality.parquet")
#df_punctuality = df_punctuality.merge(stops[['stop_id', 'stop_name']], how='left')
df_punctuality


In [ ]:
df_punctuality.columns


In [ ]:
def extract_hour(time_str):
    """Convert HH:MM:SS string to hour (0-23), with 24:00:00 = 0"""
    if pd.isna(time_str):
        return None

    parts = time_str.split(':')
    hour = int(parts[0])

    # Convert 24h to 0h (midnight)
    # if hour >= 24:
    #     hour = hour % 24
    # print(hour, minute, second)
    return hour


In [ ]:
vehicle_positions = pd.read_csv("data/all_vehicle_positions.csv" , low_memory = False , dtype={'lineId' :'object', 'pointId' : 'object'})

vehicle_positions.rename(columns={'pointId' : 'stop_id', 'lineId' : 'route_short_name', 'direction': 'direction_id'}, inplace= True)

# same direction_id {1,2} --> {1,0}
direction_id_map = {1:1, 2:0}
vehicle_positions['direction_id'] = vehicle_positions['direction_id'].map(direction_id_map)

vehicle_positions = vehicle_positions.merge(stops[['stop_id', 'stop_name']], how='left', on='stop_id')

vehicle_positions['arrival_hour'] = vehicle_positions['time'].apply(extract_hour)

vehicle_positions = vehicle_positions[['geometry', 'id', 'date', 'time', 'stop_id', 'stop_name', 'route_short_name', 'direction_id', 'distanceFromPoint', 'distance', 'uuid', 'timestamp', 'collection_timestamp', 'arrival_hour', 'day_of_week']]

vehicle_positions


In [ ]:
vehicle_positions


### prepocess vehicle position


In [ ]:
"""
PREPROCESSING: DEDUPLICATE VEHICLE POSITIONS AT STOPS
=====================================================

Problem: Vehicle positions are recorded every 30-60 seconds.
When a vehicle is stationary at a stop, we get multiple consecutive
records with the same stop_id.

Solution: Keep only the LAST observation (that has distanceFromPoint == 0) for each vehicle at each stop.
This represents the actual departure time from the stop.
"""
def deduplicate_consecutive_stops(vehicle_positions, verbose=True):
    """
    Remove duplicates for CONSECUTIVE observations at the same stop.
    For each group of consecutive stops, keep the observation with the
    smallest distanceFromPoint. If multiple observations have the same
    minimum distance:
    - For the FIRST stop group of each vehicle: keep the LAST one (departure)
    - For subsequent stops: keep the FIRST one (arrival)

    Parameters:
    -----------
    vehicle_positions : pd.DataFrame
        Raw vehicle positions
    verbose : bool
        Print statistics

    Returns:
    --------
    pd.DataFrame
        Deduplicated vehicle positions
    """

    if verbose:
        print("\n" + "="*80)
        print("DEDUPLICATING CONSECUTIVE STOPS")
        print("="*80)
        print(f"Original records: {len(vehicle_positions):,}")

    # Filter only records with stop_id
    vp_with_stops = vehicle_positions[vehicle_positions['stop_id'].notna()].copy()

    # Sort by vehicle and timestamp
    vp_sorted = vp_with_stops.sort_values(['uuid', 'timestamp'])

    # Identify consecutive duplicates
    # Create a group ID that changes when stop_id changes for the same vehicle
    vp_sorted['prev_stop'] = vp_sorted.groupby('uuid')['stop_id'].shift(1)
    vp_sorted['is_new_stop'] = (vp_sorted['stop_id'] != vp_sorted['prev_stop']) | (vp_sorted['prev_stop'].isna())
    vp_sorted['stop_group'] = vp_sorted.groupby('uuid')['is_new_stop'].cumsum()

    # For each stop group, keep the appropriate observation with minimum distanceFromPoint
    def select_best_observation(group):
        min_distance = group['distanceFromPoint'].min()
        # Filter rows with minimum distance
        min_distance_rows = group[group['distanceFromPoint'] == min_distance]

        # Check if this is the first stop group for this vehicle
        stop_group_id = group['stop_group'].iloc[0]
        is_first_stop = (stop_group_id == 1)

        if is_first_stop:
            # For first stop: take the LAST observation (departure)
            return min_distance_rows.iloc[-1]
        else:
            # For subsequent stops: take the FIRST observation (arrival)
            return min_distance_rows.iloc[0]

    vp_dedup = vp_sorted.groupby(['uuid', 'stop_group'], as_index=False).apply(select_best_observation)

    # Remove the groupby index if it was added
    if 'uuid' in vp_dedup.columns and vp_dedup.index.name == 'uuid':
        vp_dedup = vp_dedup.reset_index(drop=True)

    # Clean up helper columns
    vp_dedup = vp_dedup.drop(columns=['prev_stop', 'is_new_stop', 'stop_group'])

    if verbose:
        print(f"After deduplication: {len(vp_dedup):,}")
        print(f"Removed duplicates: {len(vp_with_stops) - len(vp_dedup):,} ({(len(vp_with_stops) - len(vp_dedup))/len(vp_with_stops)*100:.1f}%)")

        # Additional statistics
        avg_distance = vp_dedup['distanceFromPoint'].mean()
        median_distance = vp_dedup['distanceFromPoint'].median()
        print(f"\nDistance from stop statistics:")
        print(f"  Average: {avg_distance:.1f} meters")
        print(f"  Median: {median_distance:.1f} meters")

    # Sort by timestamp
    vp_dedup = vp_dedup.sort_values(['uuid', 'timestamp']).reset_index(drop=True)

    return vp_dedup


def analyze_stop_durations(vehicle_positions, sample_size=10):
    """
    Analyze how long vehicles stay at stops (time between first and last observation).

    Parameters:
    -----------
    vehicle_positions : pd.DataFrame
        Raw vehicle positions
    sample_size : int
        Number of examples to show

    Returns:
    --------
    pd.DataFrame
        Statistics about stop durations
    """

    print("\n" + "="*80)
    print("ANALYZING STOP DURATIONS")
    print("="*80)

    # Filter records with stop_id
    vp_with_stops = vehicle_positions[vehicle_positions['stop_id'].notna()].copy()

    # Calculate time at each stop for each vehicle
    stop_durations = vp_with_stops.groupby(['uuid', 'stop_id']).agg({
        'timestamp': ['min', 'max', 'count']
    }).reset_index()

    stop_durations.columns = ['uuid', 'stop_id', 'first_timestamp', 'last_timestamp', 'n_observations']
    stop_durations['duration_seconds'] = stop_durations['last_timestamp'] - stop_durations['first_timestamp']
    stop_durations['duration_minutes'] = stop_durations['duration_seconds'] / 60

    # Filter only stops with multiple observations
    multi_obs = stop_durations[stop_durations['n_observations'] > 1]

    print(f"\nTotal vehicle-stop pairs: {len(stop_durations):,}")
    print(f"Stops with multiple observations: {len(multi_obs):,} ({len(multi_obs)/len(stop_durations)*100:.1f}%)")

    if len(multi_obs) > 0:
        print(f"\nDuration statistics (for stops with multiple observations):")
        print(f"  Mean duration:   {multi_obs['duration_minutes'].mean():.2f} minutes")
        print(f"  Median duration: {multi_obs['duration_minutes'].median():.2f} minutes")
        print(f"  Max duration:    {multi_obs['duration_minutes'].max():.2f} minutes")
        print(f"  Mean observations per stop: {multi_obs['n_observations'].mean():.1f}")

        print(f"\nExamples of stops with longest durations:")
        longest = multi_obs.nlargest(sample_size, 'duration_minutes')
        print(longest[['uuid', 'stop_id', 'n_observations', 'duration_minutes']].to_string(index=False))

    return stop_durations


In [ ]:
print("="*80)
print("VEHICLE POSITION PREPROCESSING")
print("="*80)

# Analyze raw data
print("\n[1] ANALYZING RAW DATA")
stop_durations = analyze_stop_durations(vehicle_positions, sample_size=10)

# Preprocess: Simple method (recommended)
print("\n[2] PREPROCESSING DATA")
vehicle_positions_clean = deduplicate_consecutive_stops(
    vehicle_positions,
    verbose=True
)

# Save preprocessed data
print("\n[3] SAVING PREPROCESSED DATA")
#vehicle_positions_clean.to_parquet('data/vehicle_positions_clean1.parquet', index=False)
print(f"✓ Saved {len(vehicle_positions_clean):,} records to vehicle_positions_clean.parquet")

# Show example of deduplication
print("\n[4] EXAMPLE: Before vs After")
# Pick a vehicle-stop pair with multiple observations
example_uuid = stop_durations[stop_durations['n_observations'] > 3].iloc[0]['uuid']
example_stop = stop_durations[stop_durations['n_observations'] > 3].iloc[0]['stop_id']

print(f"\nVehicle: {example_uuid}")
print(f"Stop: {example_stop}")

print("\nBefore deduplication:")
before = vehicle_positions[
    (vehicle_positions['uuid'] == example_uuid) & 
    (vehicle_positions['stop_id'] == example_stop)
].sort_values('timestamp')[['uuid', 'stop_id', 'timestamp', 'date', 'time']]
print(before.to_string(index=False))

print("\nAfter deduplication:")
after = vehicle_positions_clean[
    (vehicle_positions_clean['uuid'] == example_uuid) & 
    (vehicle_positions_clean['stop_id'] == example_stop)
][['uuid', 'stop_id', 'timestamp', 'date', 'time']]
print(after.to_string(index=False))

print("\n" + "="*80)
print("PREPROCESSING COMPLETE")
print("="*80)
print(f"""
Summary:
- Original records: {len(vehicle_positions):,}
- After deduplication: {len(vehicle_positions_clean):,}
- Reduction: {len(vehicle_positions) - len(vehicle_positions_clean):,} records ({(len(vehicle_positions) - len(vehicle_positions_clean))/len(vehicle_positions)*100:.1f}%)

Next step:
Use vehicle_positions_clean instead of vehicle_positions for delay calculation.
This will significantly speed up the matching process!
""")


In [ ]:
vehicle_positions[vehicle_positions.uuid.isin(['00013ec3-c300-4712-9958-a7e10a10976e']) ]


In [ ]:
vehicle_positions_clean[vehicle_positions_clean.uuid == '00013ec3-c300-4712-9958-a7e10a10976e']


In [ ]:
vehicle_positions_clean = pd.read_parquet('data/vehicle_positions_clean1.parquet')
vehicle_positions_clean


In [ ]:
vehicle_positions_clean.columns


In [ ]:
print(f" Numbers line before deduplication {vehicle_positions['route_short_name'].nunique()}")
print(f" Numbers line after deduplication {vehicle_positions_clean['route_short_name'].nunique()}")


#### DELAY CALCULATION FOR PUNCTUALITY ANALYSIS
==========================================

This module calculates delays by matching real vehicle positions with scheduled arrivals.

Inputs:
- vehicle_positions: Real-time positions with uuid, route_short_name, stop_id, timestamp, date, time
- df_punctuality: Scheduled data with trip_id, stop_id, arrival_time_obj, route_short_name, date

Output:
- DataFrame with delays for each vehicle at each stop


### all delay optim


In [ ]:
"""
OPTIMIZED DELAY CALCULATION FOR LARGE DATASETS WITH VALIDATION
===============================================================

Memory-efficient approach using chunking and match quality validation
"""


def time_to_seconds(time_obj):
    """Convert time object to seconds since midnight."""
    if pd.isna(time_obj):
        return None
    if isinstance(time_obj, dt.time):
        return time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second
    return None


def timestamp_to_time_seconds(timestamp):
    """Convert Unix timestamp to time in seconds since midnight."""
    if pd.isna(timestamp):
        return None
    dt_obj = datetime.fromtimestamp(timestamp)
    return dt_obj.hour * 3600 + dt_obj.minute * 60 + dt_obj.second


def convert_to_time(time_str):
    """Convert HH:MM:SS string to time object."""
    if pd.isna(time_str):
        return None
    parts = time_str.split(':')
    hour = int(parts[0])
    minute = int(parts[1])
    second = int(parts[2])
    if hour >= 24:
        hour = hour % 24
    return time(hour, minute, second)


def process_chunk_with_validation(vehicles_chunk, schedule_chunk, time_tolerance_seconds, strict_matching=True):
    """
    Process a single chunk with validation to ensure match quality.
    """

    # Merge on route, direction, stop, date
    merged = vehicles_chunk.merge(
        schedule_chunk,
        on=['route_short_name', 'direction_id', 'stop_id', 'date_str'],
        how='inner',
        suffixes=('_actual', '_scheduled')
    )

    if len(merged) == 0:
        return pd.DataFrame()

    # Calculate time difference
    merged['time_diff_seconds'] = merged['actual_time_seconds'] - merged['scheduled_time_seconds']

    # Handle day wraparound
    merged.loc[merged['time_diff_seconds'] > 12*3600, 'time_diff_seconds'] -= 24*3600
    merged.loc[merged['time_diff_seconds'] < -12*3600, 'time_diff_seconds'] += 24*3600

    # Filter by time tolerance
    matched = merged[merged['time_diff_seconds'].abs() <= time_tolerance_seconds].copy()

    if len(matched) == 0:
        return pd.DataFrame()

    # Add match quality indicator
    matched['abs_diff'] = matched['time_diff_seconds'].abs()
    matched['match_quality'] = 'questionable'
    matched.loc[matched['abs_diff'] <= 120, 'match_quality'] = 'good'  # ±2 min
    matched.loc[(matched['abs_diff'] > 120) & (matched['abs_diff'] <= 180), 'match_quality'] = 'acceptable'  # 2-3 min

    # If strict matching, prioritize good/acceptable matches
    if strict_matching:
        good_matches = matched[matched['match_quality'].isin(['good', 'acceptable'])]
        if len(good_matches) > 0:
            vehicle_stop_groups = good_matches.groupby(['uuid', 'stop_id'])
            filtered_matches = []
            for name, group in vehicle_stop_groups:
                good_in_group = group[group['match_quality'] == 'good']
                if len(good_in_group) > 0:
                    filtered_matches.append(good_in_group)
                else:
                    filtered_matches.append(group)

            if len(filtered_matches) > 0:
                matched = pd.concat(filtered_matches, ignore_index=True)

    return matched


def match_vehicle_to_trip_chunked(vehicle_positions, df_punctuality, 
                                   time_tolerance_seconds=180,
                                   strict_matching=True,
                                   chunk_by='date_route_hour',
                                   hour_tolerance=1):
    """
    Memory-efficient matching by processing data in chunks with improved accuracy.
    """

    print("\n" + "="*80)
    print("IMPROVED MATCHING WITH VALIDATION")
    print("="*80)
    print(f"Strategy: {chunk_by}")
    print(f"Time tolerance: ±{time_tolerance_seconds}s ({time_tolerance_seconds/60:.1f} min)")
    print(f"Strict matching: {strict_matching}")

    # Filter only vehicle positions at stops
    vehicles_at_stops = vehicle_positions[vehicle_positions['stop_id'].notna()].copy()
    print(f"\nVehicle positions at stops: {len(vehicles_at_stops):,}")

    if len(vehicles_at_stops) == 0:
        return pd.DataFrame()

    # Pre-compute time conversions
    print("Pre-computing time conversions...")
    vehicles_at_stops['actual_time_seconds'] = vehicles_at_stops['timestamp'].apply(timestamp_to_time_seconds)
    vehicles_at_stops['date_str'] = vehicles_at_stops['date'].astype(str)
    vehicles_at_stops['actual_hour'] = (vehicles_at_stops['actual_time_seconds'] // 3600) % 24

    df_punctuality_copy = df_punctuality.copy()

    if 'scheduled_time_seconds' not in df_punctuality_copy.columns:
        print("Converting scheduled times...")
        #if df_punctuality_copy['arrival_time_obj'].dtype == 'object':
            #df_punctuality_copy['arrival_time_obj'] = df_punctuality_copy['arrival_time_obj'].apply(convert_to_time)
        df_punctuality_copy['scheduled_time_seconds'] = df_punctuality_copy['arrival_time_obj'].apply(time_to_seconds)

    df_punctuality_copy['date_str'] = df_punctuality_copy['date'].astype(str)

    if 'arrival_hour' not in df_punctuality_copy.columns:
        df_punctuality_copy['scheduled_hour'] = (df_punctuality_copy['scheduled_time_seconds'] // 3600) % 24
    else:
        df_punctuality_copy['scheduled_hour'] = df_punctuality_copy['arrival_hour']

    # Create chunks
    if chunk_by == 'date_route_hour':
        vehicles_at_stops['chunk_key'] = (vehicles_at_stops['date_str'] + '_' + 
                                          vehicles_at_stops['route_short_name'] + '_' + 
                                          vehicles_at_stops['actual_hour'].astype(str))
        df_punctuality_copy['chunk_key'] = (df_punctuality_copy['date_str'] + '_' + 
                                             df_punctuality_copy['route_short_name'] + '_' + 
                                             df_punctuality_copy['scheduled_hour'].astype(str))
        vehicle_groups = vehicles_at_stops.groupby('chunk_key')
        schedule_groups = df_punctuality_copy.groupby('chunk_key')
        group_keys = vehicles_at_stops['chunk_key'].unique()

    print(f"\nTotal chunks to process: {len(group_keys)}")

    # Process each chunk
    all_matches = []
    quality_stats = {'good': 0, 'acceptable': 0, 'questionable': 0}

    for chunk_key in tqdm(group_keys, desc="Processing chunks"):
        try:
            vehicles_chunk = vehicle_groups.get_group(chunk_key)

            if chunk_key not in schedule_groups.groups:
                continue

            schedule_chunk = schedule_groups.get_group(chunk_key)

            chunk_matches = process_chunk_with_validation(
                vehicles_chunk, 
                schedule_chunk, 
                time_tolerance_seconds,
                strict_matching
            )

            if len(chunk_matches) > 0:
                all_matches.append(chunk_matches)
                quality_stats['good'] += (chunk_matches['match_quality'] == 'good').sum()
                quality_stats['acceptable'] += (chunk_matches['match_quality'] == 'acceptable').sum()
                quality_stats['questionable'] += (chunk_matches['match_quality'] == 'questionable').sum()

            if len(all_matches) % 100 == 0:
                gc.collect()

        except Exception as e:
            print(f"\nError processing chunk {chunk_key}: {e}")
            continue

    print(f"\n✓ Processed {len(group_keys)} chunks")
    print(f"  Total matches: {sum(quality_stats.values()):,}")
    print(f"\n📊 Match Quality Distribution:")
    print(f"  Good matches (±2 min): {quality_stats['good']:,}")
    print(f"  Acceptable (2-3 min): {quality_stats['acceptable']:,}")
    print(f"  Questionable (>3 min): {quality_stats['questionable']:,}")

    if len(all_matches) == 0:
        return pd.DataFrame()

    # Combine all matches
    print("\nCombining results...")
    final_matches = pd.concat(all_matches, ignore_index=True)

    # Keep best match for each vehicle-stop observation
    print("Keeping best matches...")
    final_matches['abs_time_diff'] = final_matches['time_diff_seconds'].abs()
    best_matches = final_matches.loc[
        final_matches.groupby(['uuid', 'timestamp', 'stop_id'])['abs_time_diff'].idxmin()
    ]

    # Calculate delays
    best_matches['delay_seconds'] = best_matches['time_diff_seconds']
    best_matches['delay_minutes'] = best_matches['delay_seconds'] / 60

    print(f"\n✓ Final matches: {len(best_matches):,}")
    print(f"  Unique vehicles: {best_matches['uuid'].nunique()}")
    print(f"  Unique trips: {best_matches['trip_id'].nunique()}")

    return best_matches


def calculate_delays_optimized(vehicle_positions, df_punctuality, 
                                time_tolerance_minutes=3,
                                min_stops_per_vehicle=2,
                                chunk_by='date_route_hour',
                                strict_matching=True):
    """
    Memory-efficient delay calculation with match validation.

    Parameters:
    -----------
    vehicle_positions : pd.DataFrame
        Real vehicle positions
    df_punctuality : pd.DataFrame
        Scheduled data
    time_tolerance_minutes : float, default=3
        Time tolerance for matching (RECOMMENDED: 2-3 minutes)
    min_stops_per_vehicle : int
        Minimum stops to keep a vehicle
    chunk_by : str
        Chunking strategy
    strict_matching : bool, default=True
        Prioritize high-quality matches

    Returns:
    --------
    pd.DataFrame
        Delays dataframe
    """

    print("\n" + "="*80)
    print("OPTIMIZED DELAY CALCULATION")
    print("="*80)
    print(f"Vehicle positions: {len(vehicle_positions):,}")
    print(f"Scheduled trips: {len(df_punctuality):,}")
    print(f"Time tolerance: ±{time_tolerance_minutes} minutes")
    print(f"Strict matching: {strict_matching}")

    # Match vehicles to trips
    matches = match_vehicle_to_trip_chunked(
        vehicle_positions,
        df_punctuality,
        time_tolerance_seconds=int(time_tolerance_minutes * 60),
        strict_matching=strict_matching,
        chunk_by=chunk_by,
        hour_tolerance=1
    )

    if len(matches) == 0:
        print("\nNo matches found!")
        return pd.DataFrame()

    # Convert timestamp to arrival time
    print("\nConverting timestamps...")
    dt_utc = pd.to_datetime(matches['timestamp'], unit="s", utc=True)
    df_dt = dt_utc.dt.tz_convert("Europe/Brussels")
    matches['arrival_time'] = df_dt.dt.strftime("%H:%M:%S")

    # Select relevant columns
    delay_columns = [
        'uuid', 'trip_id', 'route_short_name', 'direction_id', 'trip_headsign',
        'stop_id', 'stop_name_actual', 'stop_sequence', 'date_str',
        'arrival_time_obj', 'arrival_time', 'day_of_week', 'timestamp',
        'actual_time_seconds', 'scheduled_time_seconds',
        'delay_seconds', 'delay_minutes', 'arrival_hour_actual',
        'match_quality'
    ]

    available_columns = [col for col in delay_columns if col in matches.columns]
    delays_df = matches[available_columns].copy()

    delays_df = delays_df.rename(columns={
        'date_str': 'date',
        'arrival_time_obj': 'scheduled_time'
    })

    # Filter by minimum stops
    print(f"\nFiltering vehicles with ≥{min_stops_per_vehicle} stops...")
    stops_per_vehicle = delays_df.groupby('uuid').size()
    valid_vehicles = stops_per_vehicle[stops_per_vehicle >= min_stops_per_vehicle].index
    delays_df = delays_df[delays_df['uuid'].isin(valid_vehicles)]

    print(f"✓ Valid vehicles: {len(valid_vehicles):,}")
    print(f"✓ Total delay records: {len(delays_df):,}")

    delays_df = delays_df.sort_values(['uuid', 'timestamp']).reset_index(drop=True)

    # Print match quality
    if 'match_quality' in delays_df.columns:
        print(f"\n📊 Final Match Quality:")
        quality_dist = delays_df['match_quality'].value_counts()
        for quality, count in quality_dist.items():
            pct = count / len(delays_df) * 100
            print(f"  {quality:15s}: {count:7,} ({pct:5.1f}%)")

    # Print statistics
    print("\n" + "="*80)
    print("DELAY SUMMARY STATISTICS")
    print("="*80)

    print(f"\nOverall Delay Statistics:")
    print(f"  Mean delay:     {delays_df['delay_minutes'].mean():7.2f} minutes")
    print(f"  Median delay:   {delays_df['delay_minutes'].median():7.2f} minutes")
    print(f"  Std deviation:  {delays_df['delay_minutes'].std():7.2f} minutes")
    print(f"  Min delay:      {delays_df['delay_minutes'].min():7.2f} minutes")
    print(f"  Max delay:      {delays_df['delay_minutes'].max():7.2f} minutes")
    print(f"  Q25:            {delays_df['delay_minutes'].quantile(0.25):7.2f} minutes")
    print(f"  Q75:            {delays_df['delay_minutes'].quantile(0.75):7.2f} minutes")

    # On-time performance (±1 minutes)
    on_time = (delays_df['delay_minutes'].abs() <= 1).sum()
    early = (delays_df['delay_minutes'] < -1).sum()
    late = (delays_df['delay_minutes'] > 1).sum()

    print(f"\nOn-Time Performance (±2 minutes):")
    print(f"  On-time:  {on_time:6,} ({on_time/len(delays_df)*100:5.2f}%)")
    print(f"  Early:    {early:6,} ({early/len(delays_df)*100:5.2f}%)")
    print(f"  Late:     {late:6,} ({late/len(delays_df)*100:5.2f}%)")

    return delays_df


In [ ]:
print("="*80)
print("OPTIMIZED DELAY CALCULATION - USAGE")
print("="*80)

# Recommended approach for large datasets
delays = calculate_delays_optimized(
    vehicle_positions_clean,
    df_punctuality,
    time_tolerance_minutes=10, 
    min_stops_per_vehicle=2,
    chunk_by='date_route_hour',
    strict_matching=False
)

# Save results
if len(delays) > 0:
    #delays.to_parquet('data/vehicle_delays_punc.parquet', index=False)
    delays.to_parquet('data/vehicle_delays_punc1.parquet', index=False)
    print(f"\n✓ Saved {len(delays):,} delay records to vehicle_delays.parquet")


In [ ]:
delays


In [ ]:
def analyze_delays_by_vehicle(delays_df):
    """
    Analyze delays at the vehicle level (trip level).

    Parameters:
    -----------
    delays_df : pd.DataFrame
        Delays dataframe from calculate_delays

    Returns:
    --------
    pd.DataFrame
        Vehicle-level statistics
    """

    print("\n" + "="*80)
    print("VEHICLE-LEVEL DELAY ANALYSIS")
    print("="*80)

    vehicle_stats = delays_df.groupby(['uuid', 'trip_id', 'route_short_name', 
                                       'direction_id', 'date']).agg({
        'delay_minutes': ['mean', 'median', 'std', 'min', 'max', 'count'],
        'stop_sequence': ['min', 'max']
    }).reset_index()

    vehicle_stats.columns = ['uuid', 'trip_id', 'route_short_name', 'direction_id', 'date',
                            'mean_delay', 'median_delay', 'std_delay', 'min_delay', 
                            'max_delay', 'n_stops', 'first_stop_seq', 'last_stop_seq']

    # Calculate delay evolution (last stop - first stop)
    first_stop_delays = delays_df.loc[
        delays_df.groupby('uuid')['stop_sequence'].idxmin()
    ][['uuid', 'delay_minutes']].rename(columns={'delay_minutes': 'first_stop_delay'})

    last_stop_delays = delays_df.loc[
        delays_df.groupby('uuid')['stop_sequence'].idxmax()
    ][['uuid', 'delay_minutes']].rename(columns={'delay_minutes': 'last_stop_delay'})

    vehicle_stats = vehicle_stats.merge(first_stop_delays, on='uuid', how='left')
    vehicle_stats = vehicle_stats.merge(last_stop_delays, on='uuid', how='left')

    vehicle_stats['delay_change'] = vehicle_stats['last_stop_delay'] - vehicle_stats['first_stop_delay']

    print(f"\nTotal vehicles analyzed: {len(vehicle_stats)}")
    print(f"\nVehicle delay statistics:")
    print(vehicle_stats[['mean_delay', 'median_delay', 'n_stops']].describe().round(2))

    print(f"\nDelay evolution (first to last stop):")
    print(f"  Improved (negative change): {(vehicle_stats['delay_change'] < -1).sum()}")
    print(f"  Stable (±1 min):            {(vehicle_stats['delay_change'].abs() <= 1).sum()}")
    print(f"  Worsened (positive change): {(vehicle_stats['delay_change'] > 1).sum()}")

    return vehicle_stats


def get_delays_for_route(delays_df, route_short_name, direction_id=None, date=None):
    """
    Filter delays for a specific route and optionally direction/date.

    Parameters:
    -----------
    delays_df : pd.DataFrame
        Delays dataframe
    route_short_name : str
        Route identifier (e.g., '5')
    direction_id : int, optional
        Direction (0 or 1)
    date : str, optional
        Specific date

    Returns:
    --------
    pd.DataFrame
        Filtered delays
    """
    filtered = delays_df[delays_df['route_short_name'] == route_short_name].copy()

    if direction_id is not None:
        filtered = filtered[filtered['direction_id'] == direction_id]

    if date is not None:
        filtered = filtered[filtered['date'] == str(date)]

    print(f"\nFiltered delays for route {route_short_name}:")
    print(f"  Total records: {len(filtered)}")
    print(f"  Unique vehicles: {filtered['uuid'].nunique()}")
    print(f"  Mean delay: {filtered['delay_minutes'].mean():.2f} min")

    return filtered


In [ ]:
vehicle_stats = analyze_delays_by_vehicle(delays)
vehicle_stats


In [ ]:
vehicle_positions[vehicle_positions.uuid == '000bffb5-c8da-4ad2-915c-38ca4fe259c4']


In [ ]:
vehicle_positions_clean[vehicle_positions_clean.uuid == '000bffb5-c8da-4ad2-915c-38ca4fe259c4']


In [ ]:
delays[delays.uuid == '0218b114-750a-4d60-9ffa-45d52a210c12']


In [ ]:
delays[delays.uuid == '000bffb5-c8da-4ad2-915c-38ca4fe259c4']


In [ ]:
delay_m6 = get_delays_for_route(delays,route_short_name='6')
delay_m6


In [ ]:
delays[delays.stop_name_actual == 'SAINT-GERY']


### Ponct Stat


In [ ]:
import json


In [ ]:
"""
GENERATE PUNCTUALITY STATISTICS FOR VISUALIZATION
=================================================

This script generates all statistics needed for the punctuality dashboard
from the delays DataFrame.

Input: delays DataFrame with columns:
- uuid, trip_id, route_short_name, direction_id, trip_headsign
- stop_id, stop_name_actual, stop_sequence
- date, day_of_week, arrival_hour_actual
- delay_seconds, delay_minutes
- scheduled_time, arrival_time

Output: Dictionary with all statistics ready for visualization
"""

def round_floats_in_obj(obj, ndigits=4):
    """
    Parcourt récursivement un dict / list et arrondit tous les floats.
    """
    if isinstance(obj, dict):
        return {k: round_floats_in_obj(v, ndigits) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [round_floats_in_obj(v, ndigits) for v in obj]
    elif isinstance(obj, float):
        return round(obj, ndigits)
    else:
        return obj


def generate_punctuality_stats(delays, on_time_threshold_min=1.0):
    """
    Generate comprehensive punctuality statistics from delays DataFrame.

    Parameters:
    -----------
    delays : pd.DataFrame
        Delays DataFrame from calculate_delays_optimized
    on_time_threshold_min : float, default=1.0
        Threshold for on-time performance (±1 minute)

    Returns:
    --------
    dict
        Dictionary with all statistics for visualization
    """

    print("\n" + "="*80)
    print("GENERATING PUNCTUALITY STATISTICS")
    print("="*80)
    print(f"Input data: {len(delays):,} delay observations")
    print(f"On-time threshold: ±{on_time_threshold_min} minutes")

    stats = {}

    # ========================================================================
    # 1. OVERALL STATISTICS
    # ========================================================================
    print("\n[1] Calculating overall statistics...")

    stats['overall'] = {
        'mean_delay': float(delays['delay_minutes'].mean()),
        'median_delay': float(delays['delay_minutes'].median()),
        'std_delay': float(delays['delay_minutes'].std()),
        'min_delay': float(delays['delay_minutes'].min()),
        'max_delay': float(delays['delay_minutes'].max()),
        'q25_delay': float(delays['delay_minutes'].quantile(0.25)),
        'q75_delay': float(delays['delay_minutes'].quantile(0.75)),
        'q90_delay': float(delays['delay_minutes'].quantile(0.90)),
        'q95_delay': float(delays['delay_minutes'].quantile(0.95)),
        'on_time_pct': float((delays['delay_minutes'].abs() <= on_time_threshold_min).sum() / len(delays) * 100),
        'early_pct': float((delays['delay_minutes'] < -on_time_threshold_min).sum() / len(delays) * 100),
        'late_pct': float((delays['delay_minutes'] > on_time_threshold_min).sum() / len(delays) * 100),
        'total_observations': int(len(delays)),
        'unique_vehicles': int(delays['uuid'].nunique()),
        'unique_trips': int(delays['trip_id'].nunique()),
        'unique_stops': int(delays['stop_id'].nunique()),
        'unique_routes': int(delays['route_short_name'].nunique())
    }

    print(f"  Mean delay: {stats['overall']['mean_delay']:.2f} min")
    print(f"  On-time rate: {stats['overall']['on_time_pct']:.1f}%")

    # ========================================================================
    # 2. DELAY DISTRIBUTION BY CATEGORY
    # ========================================================================
    print("\n[2] Calculating delay distribution by category...")

    def classify_delay(delay_min):
        if delay_min < -5:
            return 'Very Early (< -5 min)'
        elif delay_min < -on_time_threshold_min:
            return f'Early (-5 to -{on_time_threshold_min} min)'
        elif delay_min <= on_time_threshold_min:
            return f'On-Time (±{on_time_threshold_min} min)'
        elif delay_min <= 5:
            return f'Slightly Late ({on_time_threshold_min}-5 min)'
        elif delay_min <= 10:
            return 'Late (5-10 min)'
        else:
            return 'Very Late (> 10 min)'

    delays['delay_category'] = delays['delay_minutes'].apply(classify_delay)

    delay_dist = delays['delay_category'].value_counts()
    delay_dist_pct = (delay_dist / len(delays) * 100).round(1)

    category_order = [
        'Very Early (< -5 min)',
        f'Early (-5 to -{on_time_threshold_min} min)',
        f'On-Time (±{on_time_threshold_min} min)',
        f'Slightly Late ({on_time_threshold_min}-5 min)',
        'Late (5-10 min)',
        'Very Late (> 10 min)'
    ]

    stats['delayDistribution'] = []
    for cat in category_order:
        if cat in delay_dist.index:
            stats['delayDistribution'].append({
                'category': cat,
                'count': int(delay_dist[cat]),
                'pct': float(delay_dist_pct[cat])
            })

    print(f"  Categories generated: {len(stats['delayDistribution'])}")

    # ========================================================================
    # 3. STATISTICS BY ROUTE
    # ========================================================================
    print("\n[3] Calculating statistics by route...")

    route_groups = delays.groupby('route_short_name')

    route_stats_list = []
    for route, group in route_groups:
        route_stats_list.append({
            'route': str(route),
            'mean_delay': float(group['delay_minutes'].mean()),
            'median_delay': float(group['delay_minutes'].median()),
            'std_delay': float(group['delay_minutes'].std()),
            'on_time_pct': float((group['delay_minutes'].abs() <= on_time_threshold_min).sum() / len(group) * 100),
            'count': int(len(group)),
            'unique_vehicles': int(group['uuid'].nunique())
        })

    # Sort by mean delay (worst first for easy identification)
    stats['routeStats'] = sorted(route_stats_list, key=lambda x: x['mean_delay'], reverse=True)

    print(f"  Routes analyzed: {len(stats['routeStats'])}")

    # ========================================================================
    # 4. HOURLY PATTERN
    # ========================================================================
    print("\n[4] Calculating hourly pattern...")

    if 'arrival_hour_actual' in delays.columns:
        hour_col = 'arrival_hour_actual'
    elif 'arrival_hour' in delays.columns:
        hour_col = 'arrival_hour'
    else:
        print("  Warning: No hour column found, skipping hourly analysis")
        stats['hourlyPattern'] = []
        hour_col = None

    if hour_col:
        hourly_groups = delays.groupby(hour_col)

        hourly_stats_list = []
        for hour, group in hourly_groups:
            if pd.notna(hour):
                hourly_stats_list.append({
                    'hour': int(hour),
                    'mean_delay': float(group['delay_minutes'].mean()),
                    'median_delay': float(group['delay_minutes'].median()),
                    'on_time_pct': float((group['delay_minutes'].abs() <= on_time_threshold_min).sum() / len(group) * 100),
                    'count': int(len(group))
                })

        stats['hourlyPattern'] = sorted(hourly_stats_list, key=lambda x: x['hour'])
        print(f"  Hours analyzed: {len(stats['hourlyPattern'])}")

    # ========================================================================
    # 5. DAY OF WEEK PATTERN
    # ========================================================================
    print("\n[5] Calculating day of week pattern...")

    if 'day_of_week' in delays.columns:
        day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

        day_groups = delays.groupby('day_of_week')

        day_stats_dict = {}
        for day, group in day_groups:
            day_stats_dict[day] = {
                'day': day,
                'mean_delay': float(group['delay_minutes'].mean()),
                'median_delay': float(group['delay_minutes'].median()),
                'on_time_pct': float((group['delay_minutes'].abs() <= on_time_threshold_min).sum() / len(group) * 100),
                'count': int(len(group))
            }

        # Sort by day order
        stats['dayOfWeekPattern'] = [day_stats_dict[day] for day in day_order if day in day_stats_dict]
        print(f"  Days analyzed: {len(stats['dayOfWeekPattern'])}")
    else:
        print("  Warning: No day_of_week column found")
        stats['dayOfWeekPattern'] = []

    # ========================================================================
    # 6. STOP ANALYSIS (TOP WORST AND BEST)
    # ========================================================================
    print("\n[6] Calculating stop statistics...")

    if 'stop_name_actual' in delays.columns:
        stop_col = 'stop_name_actual'
    elif 'stop_name' in delays.columns:
        stop_col = 'stop_name'
    else:
        print("  Warning: No stop name column found")
        stats['topWorstStops'] = []
        stats['topBestStops'] = []
        stop_col = None

    if stop_col:
        # Group by both stop name and route
        stop_groups = delays.groupby([stop_col, 'route_short_name'])

        stop_stats_list = []
        for (stop_name, route_short_name), group in stop_groups:
            # Only include stops with at least 20 observations for reliability
            if len(group) >= 20:
                stop_stats_list.append({
                    'stop_name': str(stop_name),
                    'route': str(route_short_name),
                    'mean_delay': float(group['delay_minutes'].mean()),
                    'median_delay': float(group['delay_minutes'].median()),
                    'on_time_pct': float((group['delay_minutes'].abs() <= on_time_threshold_min).sum() / len(group) * 100),
                    'count': int(len(group))
                })

        # Sort by mean delay
        sorted_stops = sorted(stop_stats_list, key=lambda x: x['mean_delay'], reverse=True)

        stats['topWorstStops'] = sorted_stops[:10]  # Top 10 worst
        stats['topBestStops'] = sorted_stops[-10:][::-1]  # Top 10 best (reversed)

        print(f"  Total stop-route combinations analyzed: {len(stop_stats_list)}")
        if stats['topWorstStops']:
            worst = stats['topWorstStops'][0]
            print(f"  Worst stop: {worst['stop_name']} (Route {worst['route']}) - {worst['mean_delay']:.2f} min")
        if stats['topBestStops']:
            best = stats['topBestStops'][0]
            print(f"  Best stop: {best['stop_name']} (Route {best['route']}) - {best['mean_delay']:.2f} min")

    # ========================================================================
    # 7. DIRECTION ANALYSIS (if available)
    # ========================================================================
    print("\n[7] Calculating direction statistics...")

    if 'direction_id' in delays.columns and 'trip_headsign' in delays.columns:
        direction_groups = delays.groupby(['route_short_name', 'direction_id', 'trip_headsign'])

        direction_stats_list = []
        for (route, direction, headsign), group in direction_groups:
            if len(group) >= 20:  # Min 10 observations
                direction_stats_list.append({
                    'route': str(route),
                    'direction_id': int(direction),
                    'headsign': str(headsign),
                    'mean_delay': float(group['delay_minutes'].mean()),
                    'on_time_pct': float((group['delay_minutes'].abs() <= on_time_threshold_min).sum() / len(group) * 100),
                    'count': int(len(group))
                })

        stats['directionStats'] = direction_stats_list
        print(f"  Route-direction pairs analyzed: {len(stats['directionStats'])}")
    else:
        stats['directionStats'] = []

    # ========================================================================
    # 8. TEMPORAL TRENDS (by date)
    # ========================================================================
    print("\n[8] Calculating temporal trends...")

    if 'date' in delays.columns:
        date_groups = delays.groupby('date')

        date_stats_list = []
        for date, group in date_groups:
            date_stats_list.append({
                'date': str(date),
                'mean_delay': float(group['delay_minutes'].mean()),
                'on_time_pct': float((group['delay_minutes'].abs() <= on_time_threshold_min).sum() / len(group) * 100),
                'count': int(len(group))
            })

        stats['temporalTrends'] = sorted(date_stats_list, key=lambda x: x['date'])
        print(f"  Dates analyzed: {len(stats['temporalTrends'])}")
    else:
        stats['temporalTrends'] = []

    print("\n" + "="*80)
    print("STATISTICS GENERATION COMPLETE")
    print("="*80)

    stats = round_floats_in_obj(stats, ndigits=4)  
    return stats


def save_stats_to_json(stats, filename='punctuality_stats.json'):
    """
    Save statistics to JSON file for use in dashboard.

    Parameters:
    -----------
    stats : dict
        Statistics dictionary from generate_punctuality_stats
    filename : str
        Output filename
    """
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Statistics saved to {filename}")


def print_summary(stats):
    """
    Print a summary of the generated statistics.

    Parameters:
    -----------
    stats : dict
        Statistics dictionary
    """
    print("\n" + "="*80)
    print("STATISTICS SUMMARY")
    print("="*80)

    print(f"\n📊 Overall Performance:")
    print(f"  Mean Delay: {stats['overall']['mean_delay']:.2f} minutes")
    print(f"  Median Delay: {stats['overall']['median_delay']:.2f} minutes")
    print(f"  On-Time Rate: {stats['overall']['on_time_pct']:.1f}%")
    print(f"  Early Rate: {stats['overall']['early_pct']:.1f}%")
    print(f"  Late Rate: {stats['overall']['late_pct']:.1f}%")

    print(f"\n🚇 Data Coverage:")
    print(f"  Total Observations: {stats['overall']['total_observations']:,}")
    print(f"  Unique Vehicles: {stats['overall']['unique_vehicles']:,}")
    print(f"  Unique Routes: {stats['overall']['unique_routes']}")
    print(f"  Unique Stops: {stats['overall']['unique_stops']:,}")

    if stats['routeStats']:
        print(f"\n🚌 Routes Analyzed: {len(stats['routeStats'])}")
        worst_route = stats['routeStats'][0]
        best_route = stats['routeStats'][-1]
        print(f"  Worst performing: Line {worst_route['route']} ({worst_route['mean_delay']:.2f} min)")
        print(f"  Best performing: Line {best_route['route']} ({best_route['mean_delay']:.2f} min)")

    if stats['topWorstStops']:
        print(f"\n📍 Stops Analyzed:")
        print(f"  Worst stop: {stats['topWorstStops'][0]['stop_name']} - (Route {stats['topWorstStops'][0]['route']}) - ({stats['topWorstStops'][0]['mean_delay']:.2f} min)")
        print(f"  Best stop: {stats['topBestStops'][0]['stop_name']}  - (Route {stats['topBestStops'][0]['route']}) -  ({stats['topBestStops'][0]['mean_delay']:.2f} min)")

    if stats['hourlyPattern']:
        peak_hours = [h for h in stats['hourlyPattern'] if h['mean_delay'] > stats['overall']['mean_delay'] * 1.5]
        if peak_hours:
            print(f"\n🕐 Peak Hours (>1.5× avg delay): {[h['hour'] for h in peak_hours]}")


In [ ]:
if __name__ == "__main__":

    print("="*80)
    print("PUNCTUALITY STATISTICS GENERATOR")
    print("="*80)

    # Generate all statistics
    punct_stats = generate_punctuality_stats(
        delays,
        on_time_threshold_min=1.0  # ±1 minute for on-time
    )

    # Print summary
    print_summary(punct_stats)

    # Save to JSON
    save_stats_to_json(punct_stats, 'stib-dashboard/stib-dashboard-punc/src/data/punctuality_stats.json')

    # Example: Access specific statistics
    print("\n" + "="*80)
    print("Accessing Statistics")
    print("="*80)

    print("\nOverall mean delay:", punct_stats['overall']['mean_delay'])
    print("On-time rate:", punct_stats['overall']['on_time_pct'])

    print("\nTop 3 worst routes:")
    for i, route in enumerate(punct_stats['routeStats'][:3], 1):
        print(f"  {i}. Line {route['route']}: {route['mean_delay']:.2f} min (OTP: {route['on_time_pct']:.1f}%)")

    print("\nTop 3 worst stops:")
    for i, stop in enumerate(punct_stats['topWorstStops'][:3], 1):
        print(f"  {i}. {stop['stop_name']}: {stop['mean_delay']:.2f} min (OTP: {stop['on_time_pct']:.1f}%)")

    print("\n" + "="*80)
    print("✓ Statistics ready for visualization!")
    print("="*80)


In [ ]:
punct_stats['overall']


In [ ]:
delays.route_short_name.nunique()


### Stats Viz
